In [1]:
import sys
sys.path.append("../src")
import numpy as np
from dataset import NeuralSequenceDataset

data = np.load("../data/processed_sub01_ses02.npz")
windows, labels = data["windows"], data["labels"]

ds = NeuralSequenceDataset(windows, labels, seq_len=16)
print("Dataset length:", len(ds))

seq, target, target_label = ds[0]
print("seq shape:", seq.shape)
print("target shape:", target.shape)
print("target_label:", target_label.item())

Dataset length: 2168
seq shape: torch.Size([16, 64, 50])
target shape: torch.Size([64, 50])
target_label: 0


In [2]:
!cat ../src/dataset.py

import torch
from torch.utils.data import Dataset

class NeuralSequenceDataset(Dataset):

    def __init__(self, windows, labels, seq_len=16):
        self.windows = torch.tensor(windows, dtype=torch.float32)  # (N, C, T)
        self.labels = torch.tensor(labels, dtype=torch.long)
        self.seq_len = seq_len

    def __len__(self):
        return len(self.windows) - self.seq_len - 1

    def __getitem__(self, idx):
        seq = self.windows[idx : idx + self.seq_len]       # (seq_len, C, T)
        target = self.windows[idx + self.seq_len]           # (C, T) - the "next token"
        target_label = self.labels[idx + self.seq_len]      # movement/rest, for eval only
        return seq, target, target_label

In [4]:
import torch
from torch.utils.data import DataLoader
from model import NeuralGPT, info_nce_loss

loader = DataLoader(ds, batch_size=8, shuffle=True)
seq_batch, target_batch, label_batch = next(iter(loader))
print(seq_batch.shape, target_batch.shape)

model = NeuralGPT(n_channels=64, window_samples=50, seq_len=16)
pred, target_emb, hidden = model(seq_batch, target_batch)
print(pred.shape, target_emb.shape)

loss = info_nce_loss(pred, target_emb)
print(loss.item())

torch.Size([8, 16, 64, 50]) torch.Size([8, 64, 50])
torch.Size([8, 128]) torch.Size([8, 128])
2.490039110183716


In [5]:
import numpy as np
from torch.utils.data import random_split

n_val = int(0.1 * len(ds))
train_ds, val_ds = random_split(ds, [len(ds) - n_val, n_val])
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32)

device = "mps" if torch.backends.mps.is_available() else "cpu"
print(device)

model = NeuralGPT(n_channels=64, window_samples=50, seq_len=16).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.01)

mps


In [6]:
EPOCHS = 30

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    for seq, target, _ in train_loader:
        seq, target = seq.to(device), target.to(device)
        pred, target_emb, _ = model(seq, target)
        loss = info_nce_loss(pred, target_emb)
        opt.zero_grad()
        loss.backward()
        opt.step()
        train_loss += loss.item()

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for seq, target, _ in val_loader:
            seq, target = seq.to(device), target.to(device)
            pred, target_emb, _ = model(seq, target)
            val_loss += info_nce_loss(pred, target_emb).item()

    print(epoch, train_loss / len(train_loader), val_loss / len(val_loader))

0 2.2541918989087715 1.3818424599511283
1 0.5588593106777942 0.9094746708869934
2 0.2500675704146995 0.7529452613421849
3 0.1596557177969667 0.6871887615748814
4 0.12203591720002596 0.6169409879616329
5 0.10033138059690351 0.5855609433991569
6 0.08502539344986931 0.5583177762372153
7 0.07157773506201681 0.5440488542829242
8 0.06600593335804392 0.5254864777837481
9 0.058652185514325 0.5156281377588
10 0.055834324816699886 0.5063700973987579
11 0.049402389553238134 0.49813781891550335
12 0.045958354947019796 0.47700187989643644
13 0.04265733072381528 0.47571760416030884
14 0.0404114769924371 0.4711593730109079
15 0.03749711835970644 0.46150403789111544
16 0.03393259366638348 0.45993711267198834
17 0.03381818600121091 0.4561370994363512
18 0.031556711364232125 0.44954281619616915
19 0.03081384191259009 0.44751464894839693
20 0.028707141758965663 0.43411578450884136
21 0.027355197878157506 0.4301200338772365
22 0.026100281320634435 0.43964398758752005
23 0.02535719729837824 0.4401363134384

In [8]:
torch.save(model.state_dict(), "../neurogpt.pt")